In [1]:
# Import potrzebnych bibliotek.
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [ ]:
# Wczytanie zmiennych środowiskowych.
load_dotenv()

# Dane potrzebne do bazy PostgreSQL.
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
name = os.getenv('DB_NAME')
port = os.getenv('DB_PORT')
host = os.getenv('DB_HOST')

# Stwarzam adres połączeniowy oraz silnik bazy danych.
db_url = f"postgresql://{user}:{password}@{host}:{port}/{name}"
engine = create_engine(db_url)

query = "SELECT * FROM credit_risk"

# Załadowanie wyniku zapytania do tabeli w Pandas.
df = pd.read_sql(query, con = engine)

# Wyświetlenie 10 pierwszych wierszy.
df.head(10)

,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,1,22,59000.0,RENT,123.0,PERSONAL,D,35000.0,16.02,1,0.59,Y,3
1,2,21,9600.0,OWN,5.0,EDUCATION,B,1000.0,11.14,0,0.10,N,2
2,3,25,9600.0,MORTGAGE,1.0,MEDICAL,C,5500.0,12.87,1,0.57,N,3
3,4,23,65500.0,RENT,4.0,MEDICAL,C,35000.0,15.23,1,0.53,N,2
4,5,24,54400.0,RENT,8.0,MEDICAL,C,35000.0,14.27,1,0.55,Y,4
5,6,21,9900.0,OWN,2.0,VENTURE,A,2500.0,7.14,1,0.25,N,2
6,7,26,77100.0,RENT,8.0,EDUCATION,B,35000.0,12.42,1,0.45,N,3
7,8,24,78956.0,RENT,5.0,MEDICAL,B,35000.0,11.11,1,0.44,N,4
8,9,24,83000.0,RENT,8.0,PERSONAL,A,35000.0,8.90,1,0.42,N,2
9,10,21,10000.0,OWN,6.0,VENTURE,D,1600.0,14.74,1,0.16,N,3


In [3]:
# Początkowy rozmiar.
df.shape

(32581, 13)

In [4]:
# Analiza rozkładu numerycznego.
df.describe()

,id,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,16291.000000,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,9405.468897,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,1.000000,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,8146.000000,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,16291.000000,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,24436.000000,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,32581.000000,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [5]:
# Sprawdzenie typów danych i ogólnej kompletności.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          32581 non-null  int64  
 1   person_age                  32581 non-null  int64  
 2   person_income               32581 non-null  float64
 3   person_home_ownership       32581 non-null  str    
 4   person_emp_length           31686 non-null  float64
 5   loan_intent                 32581 non-null  str    
 6   loan_grade                  32581 non-null  str    
 7   loan_amnt                   32581 non-null  float64
 8   loan_int_rate               29465 non-null  float64
 9   loan_status                 32581 non-null  int64  
 10  loan_percent_income         32581 non-null  float64
 11  cb_person_default_on_file   32581 non-null  str    
 12  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(5), int64(4), str(4)
memory usa

In [6]:
# Zliczenie brakujących danych.
df.isnull().sum()

id                               0
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [7]:
# Zliczenie duplikatów.
df.duplicated().sum()

np.int64(0)

In [8]:
# Zachowuje tylko klientów w wieku do 90 lat włącznie.
df = df[df['person_age'] <= 90]

# Zachowuje logiczny staż pracy (do 50 lat). Warunek or służy do ochrony wierszy, które mają wartość NaN,
df = df[(df['person_emp_length'] <= 50) | (df['person_emp_length'].isnull())]

In [9]:
# Obliczenie i aplikacja mediany stażu pracy dla klientów ze stażem pracy NaN.
emp_median = df['person_emp_length'].median()
df['person_emp_length'] = df['person_emp_length'].fillna(emp_median)

In [10]:
# Obliczenie mediany każdej z klas kredytowych.
group_medians = df.groupby('loan_grade')['loan_int_rate'].transform('median')

# Wypełnienie obliczoną medianą, dla danej klasy, osób z wartością NaN przy oprocentowaniu kredytu.
df['loan_int_rate'] = df['loan_int_rate'].fillna(group_medians)

In [11]:
# Weryfikacja braków 
df.isnull().sum()

id                            0
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [12]:
# Końcowy rozmiar, usunięto 8 wierszy z błędnymi danymi.
df.shape

(32573, 13)

In [ ]:
# Zapis wyczyszczonego pliku do formatu .csv.
df.to_csv('../dane/przetworzone_dane/credit_risk_dataset_cleaned.csv', index = False)